# V2 Workpiece Analysis (Global Chamfer)
This notebook calculates the global Chamfer Distance between the entire noisy scan and the perfect simulated raycast for every viewpoint.

In [1]:
import os
import glob
import pandas as pd
import numpy as np
import open3d as o3d
import time

# ==========================================
# 1. CONFIGURATION
# ==========================================
WORKPIECES = ["TH0011AV", "TH0012AV", "TH0021AV", "TH0022AV", "TH0031AV", "TH0032AV", "TH0041AV", "TH0042AV", "TH0051AV", "TH0052AV", "TH0061AV", "TH0062AV", "TH0071AV", "TH0072AV"]
WORKPIECES = ["TH0011AV"]
EXPERIMENT = "test_8_simulation2"
DATASET_NAME = EXPERIMENT
DISTANCE_TRESHOLD = 2.0


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
# ==========================================
# 2. EVALUATION LOOP
# ==========================================
results = []

for workpiece in WORKPIECES:
    print(f"\n==========================================")
    print(f"ANALYZING GLOBAL CHAMFER: {workpiece}")
    print(f"==========================================")
    
    SIM_DIR = f"viewpoints_candidate/testing_data/{EXPERIMENT}/{workpiece}"
    SIMULATION_OUTPUT_DIR = f"simulation/{EXPERIMENT}/{workpiece}"
    PROCESSED_DIR = f"processed_data/{EXPERIMENT}/{workpiece}"
    
    pcd_files = glob.glob(os.path.join(SIMULATION_OUTPUT_DIR, "viewpoint_simulated_noise_*.pcd"))
    if not pcd_files:
        pcd_files = glob.glob(os.path.join(PROCESSED_DIR, "viewpoint_simulated_noise_*.pcd"))
        
    pcd_files = [f for f in pcd_files if "_surface" not in f]
    if not pcd_files:
        print(f"No simulated point clouds found for {workpiece}.")
        continue
        
    start_time = time.time()
    count_hits = 0
    
    for noisy_pcd_path in pcd_files:
        viewpoint_name_noisy = os.path.basename(noisy_pcd_path).replace('.pcd', '')
        view_idx = int(viewpoint_name_noisy.replace('viewpoint_simulated_noise_', ''))
        
        perfect_pcd_path = os.path.join(SIM_DIR, f"viewpoint_simulated_{view_idx}.pcd")
        if not os.path.exists(perfect_pcd_path):
            print(f"Missing perfect CAD for view {view_idx}, skipping.")
            continue
            
        noisy_pcd = o3d.io.read_point_cloud(noisy_pcd_path)
        perfect_pcd = o3d.io.read_point_cloud(perfect_pcd_path)
        
        if len(noisy_pcd.points) == 0 or len(perfect_pcd.points) == 0:
            continue
            
        dists_noisy = np.asarray(noisy_pcd.compute_point_cloud_distance(perfect_pcd))
        dists_perfect = np.asarray(perfect_pcd.compute_point_cloud_distance(noisy_pcd))
        
        # Use threshold to filter massive outliers
        valid_noisy_indices = np.where(dists_noisy < DISTANCE_TRESHOLD)[0]
        valid_perf_indices = np.where(dists_perfect < DISTANCE_TRESHOLD)[0]
        
        if len(valid_noisy_indices) == 0 or len(valid_perf_indices) == 0:
            continue
            
        mean_s2c = np.mean(dists_noisy[valid_noisy_indices])
        mean_c2s = np.mean(dists_perfect[valid_perf_indices])
        
        global_chamfer = mean_s2c + mean_c2s
        
        results.append({
            'Workpiece': workpiece,
            'Viewpoint': view_idx,
            'Chamfer_Distance_mm': global_chamfer
        })
        count_hits += 1
        
    print(f"Processed {count_hits} viewpoints in {time.time() - start_time:.1f}s")



ANALYZING GLOBAL CHAMFER: TH0011AV
Processed 432 viewpoints in 22.5s


In [3]:
# ==========================================
# 3. EXPORT RESULTS
# ==========================================
if results:
    df_results = pd.DataFrame(results)
    
    CSV_EXPORT_DIR = f"processed_data/{DATASET_NAME}"
    os.makedirs(CSV_EXPORT_DIR, exist_ok=True)
    csv_path = os.path.join(CSV_EXPORT_DIR, "V2_chamfer_results.csv")
    
    df_results.to_csv(csv_path, index=False)
    print(f"\nSaved {len(df_results)} global chamfer scores to {csv_path}")
else:
    print("No results to save.")



Saved 432 global chamfer scores to processed_data/test_8_simulation2\V2_chamfer_results.csv


In [13]:
# ==========================================
# 4. INTERACTIVE VISUALIZATION
# ==========================================
import math
import copy

WORKPIECE_VIS = "TH0011AV"
VIEWPOINT_IDX_VIS = 1

print(f"Loading {WORKPIECE_VIS} - Viewpoint {VIEWPOINT_IDX_VIS} for interactive validation...")

SIM_DIR_VIS = f"viewpoints_candidate/testing_data/{EXPERIMENT}/{WORKPIECE_VIS}"
SIMULATION_OUTPUT_DIR_VIS = f"simulation/{EXPERIMENT}/{WORKPIECE_VIS}"

PROCESSED_DIR_VIS = f"processed_data/{EXPERIMENT}/{WORKPIECE_VIS}"
noisy_pcd_path_vis = os.path.join(SIMULATION_OUTPUT_DIR_VIS, f"viewpoint_simulated_noise_{VIEWPOINT_IDX_VIS}.pcd")
if not os.path.exists(noisy_pcd_path_vis):
    noisy_pcd_path_vis = os.path.join(PROCESSED_DIR_VIS, f"viewpoint_simulated_noise_{VIEWPOINT_IDX_VIS}.pcd")
perfect_pcd_path_vis = os.path.join(SIM_DIR_VIS, f"viewpoint_simulated_{VIEWPOINT_IDX_VIS}.pcd")

def default_visualization(geometries, window_name="Visualization", zoom=1.0):
    azimuth_deg = -45
    elevation_deg = -135
    az = math.radians(azimuth_deg)
    el = math.radians(elevation_deg)
    front = np.array([math.cos(el) * math.cos(az), math.cos(el) * math.sin(az), math.sin(el)])
    front = -front
    if isinstance(geometries, list) and len(geometries) > 0:
        lookat = geometries[0].get_center()
    else:
        lookat = [0, 0, 0]
    up = [0, 0, 1]
    o3d.visualization.draw_geometries(geometries, window_name=window_name, width=1024, height=768, lookat=lookat, up=up, front=front, zoom=zoom)

try:
    # Load Noisy PCD
    noisy_pcd_vis = o3d.io.read_point_cloud(noisy_pcd_path_vis)
    noisy_pcd_vis.paint_uniform_color([0.5, 0.5, 0.5]) # Gray for Noisy

    # Load Perfect PCD
    perfect_pcd_vis = o3d.io.read_point_cloud(perfect_pcd_path_vis)
    perfect_pcd_vis.paint_uniform_color([1, 0, 0]) # Red for Perfect

    print("\nGray: Full Noisy Simulated Viewpoint")
    print("Red: Perfect Simulated Raycast")
    
    default_visualization([noisy_pcd_vis, perfect_pcd_vis], window_name="Global Chamfer Validation (Gray=Noisy, Red=Perfect)")

except Exception as e:
    print(f"Error loading or visualizing point clouds: {e}")


Loading TH0011AV - Viewpoint 1 for interactive validation...

Gray: Full Noisy Simulated Viewpoint
Red: Perfect Simulated Raycast
